In [0]:
%pip install -U langchain langchain-google-genai langchain-openai python-dotenv openai
dbutils.library.restartPython()

In [0]:
import importlib
import mlflow
import moderate
import pandas as pd

mlflow.set_experiment("/moderation")

importlib.reload(moderate)

def run_evaluation(model, test_data):
    agent = moderate.initialize_agent(model, dbutils)

    results = []

    for description, flag in test_data:
        ground_truth = flag == "is_valid"

        result = moderate.moderate_profile_description(description, agent)

        print(result)

        prediction = result["is_valid"]

        results.append({
            "input_text": description,
            "ground_truth_flag": flag,
            "ground_truth": ground_truth,
            "prediction": prediction,
            "correct": prediction == ground_truth,
            "language": result["language"],
            "reason": result["reason"],
            "confidence": result["confidence"],
        })

    return pd.DataFrame(results)

def compute_metrics(df):
    # prediction / ground_truth are booleans:
    # True = valid, False = invalid
    y_pred = df["prediction"].astype(bool)
    y_true = df["ground_truth"].astype(bool)

    accuracy = (y_pred == y_true).mean()

    # Treat "invalid" (False) as the positive class for moderation metrics
    tp_invalid = ((y_pred == False) & (y_true == False)).sum()
    fp_invalid = ((y_pred == False) & (y_true == True)).sum()
    fn_invalid = ((y_pred == True) & (y_true == False)).sum()

    precision_invalid = tp_invalid / (tp_invalid + fp_invalid) if (tp_invalid + fp_invalid) > 0 else 0.0
    recall_invalid = tp_invalid / (tp_invalid + fn_invalid) if (tp_invalid + fn_invalid) > 0 else 0.0

    print(f"Accuracy: {accuracy}")
    print(f"Precision (invalid): {precision_invalid}")
    print(f"Recall (invalid): {recall_invalid}")

    return {
        "accuracy": float(accuracy),
        "precision_invalid": float(precision_invalid),
        "recall_invalid": float(recall_invalid),
    }

def execute_run(model, test_data):
    with mlflow.start_run():
        mlflow.log_param("model", model)

        results_df = run_evaluation(model, test_data)
        mlflow.log_table(results_df, "predictions.json")

        metrics = compute_metrics(results_df)
        mlflow.log_metrics(metrics)

In [ ]:
from get_test_data import get_random_test_data
from models import GEMINI_3_5_FLASH, GPT_5_4_NANO

test_data = get_random_test_data(3)

execute_run(GEMINI_3_5_FLASH, test_data)

execute_run(GPT_5_4_NANO, test_data)